# Modelo del recomendador de videojuegos

Este notebook genera el archivo `data/juegos.js` que usa la app. Parte de las calificaciones de videojuegos de Amazon Reviews'23, limpia el catálogo, compara tres formas de recomendar y exporta el modelo elegido: PCA de la matriz usuario-juego con 50 componentes.

Necesita dos archivos en la carpeta `datos/`, junto a este notebook:

- `Video_Games.csv.gz`: las calificaciones (versión 5-core de Amazon Reviews'23).
- `meta_juegos.csv.gz`: títulos, plataformas e imágenes, generado con la celda de Colab que viene en el README.

In [ ]:
%pip install numpy pandas scipy scikit-learn

In [7]:
import base64
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.utils.extmath import randomized_svd

SEMILLA = 69

## 1. Limpieza del catálogo

En la categoría de videojuegos de Amazon vienen mezclados juegos, accesorios, consolas, membresías y contenido descargable. Se conservaron solo los productos cuya categoría termina en "Games" y se quitaron los que, por su título, son tarjetas, membresías, paquetes o periféricos.

Además, el mismo juego aparece una vez por plataforma y edición, por ejemplo "Grand Theft Auto V: Premium Edition - Xbox One [Digital Code]". Se limpió el título (sin plataforma, sin edición, sin texto entre corchetes) y se unieron las versiones que quedan con el mismo título. Así el usuario califica "Grand Theft Auto V" una sola vez y la matriz queda menos dispersa.

In [8]:
NO_JUEGO = re.compile(
    r'membership|gift card|giftcard|\bpoints?\b|subscription|season pass|\bdlc\b|challenger pack|'
    r'xbox live|playstation plus|ps plus|game pass|currency|v-bucks|cash card|prepaid|wallet|'
    r'expansion pass|add-on|addon|\bcoins?\b|crystals|shark card|\bcredits?\b|network card|'
    r'\bstarter pack\b|\bbundle\b|\bconsole\b|pre-?paid|time card|game card|collectible case|\bsensor\b|chocolate|'
    r'candy|gourmet|\boz\b|\bcontroller\b|headset|charger|\bcable\b|memory card|\bskin\b|steering wheel|\bguitar\b(?! hero)',
    re.I)

EDICION_SUELTA = (r"(deluxe|limited|collector'?s|steelbook|standard|goty|game of the year|day one|day 1|launch|"
                  r"premium|digital|physical|platinum hits|greatest hits|player'?s choice|nintendo selects|standard game)")
EDICION_CON_PALABRA = (r"((steelbook|ultimate|complete|gold|legendary|legend|hall of fame|champions|definitive|special|legacy|signature|tricentennial|"
                       r"anniversary|enhanced|bonus|collector'?s|deluxe|limited|standard|premium|digital|"
                       r"game of the year|goty|launch|day one)\s+)+(edition|version|ed\.)")
SUFIJO_EDICION = re.compile(r"(:\s*|\s+)(" + EDICION_CON_PALABRA + r"|" + EDICION_SUELTA + r"(\s+" + EDICION_SUELTA + r")*)\s*$", re.I)

PLATAFORMAS_FINAL = re.compile(
    r"\b(ps[1-5]|ps vita|psp|playstation\s*(vita|portable|[1-5])?|xbox\s*(one|360|series\s*[xs](\s*[&/|]\s*[xs])?)?|"
    r"nintendo\s*(switch|wii u|wii|3ds|ds|64|gamecube)?|switch|wii u|wii|3ds|2ds|ds|gamecube|n64|"
    r"pc|mac|windows( 10| 7)?|steam|origin|uplay|online game code|download|code)\s*$",
    re.I)


EDITORAS = {'sega', 'ubisoft', 'nintendo', 'capcom', 'ea', 'bandai namco', 'konami', 'square enix', 'activision'}


def plataforma(cats):
    hojas = [c for c in cats if c not in ('Video Games', 'Legacy Systems', 'Games')
             and not c.endswith('Systems')]
    return hojas[-1] if hojas else 'Otra'


def titulo_base(titulo):
    """Título para mostrar: sin plataforma, sin edición y sin texto entre corchetes."""
    t = str(titulo)
    if re.search(r'kirby[: ]+triple deluxe', t, re.I):
        return 'Kirby: Triple Deluxe'
    t = re.sub(r'\[.*?\]|\(.*?\)', ' ', t)
    partes = re.split(r'\s+[-–—|]\s+', t)
    t = partes[1] if len(partes) > 1 and partes[0].strip().lower() in EDITORAS else partes[0]
    t = re.sub(r'^\s*(ps[1-5]|ps vita|wb games|nintendo selects:?|sony)\s+', '', t, flags=re.I)
    for _ in range(3):
        t2 = PLATAFORMAS_FINAL.sub('', t).strip(' -:,')
        t2 = SUFIJO_EDICION.sub('', t2).strip(' -:,')
        t2 = re.sub(r'\s+(for|hits|playstation hits|software)\s*$', '', t2, flags=re.I).strip(' -:,')
        if t2 == t:
            break
        t = t2
    t = re.sub(r'\s{2,}', ' ', t).strip(' -:,')
    return t if t else str(titulo)


def clave(titulo):
    """Clave para unir versiones del mismo juego."""
    t = unicodedata.normalize('NFKD', titulo_base(titulo)).encode('ascii', 'ignore').decode().lower()
    t = t.replace('&', ' and ')
    t = re.sub(r"[^a-z0-9 ]", ' ', t)
    t = re.sub(r'\bthe\b', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()


def anio(detalles):
    d = json.loads(detalles)
    for campo in ('Release date', 'Date First Available'):
        m = re.search(r'(19[7-9]\d|20[0-2]\d)', str(d.get(campo, '')))
        if m:
            return int(m.group(1))
    return np.nan


def cargar(ruta_meta, ruta_ratings):
    meta = pd.read_csv(ruta_meta)
    ratings = pd.read_csv(ruta_ratings)
    meta['cats'] = meta['categories'].map(json.loads)
    juegos = meta[meta['cats'].map(lambda c: len(c) > 0 and c[-1] == 'Games')].copy()
    juegos = juegos[juegos['title'].notna() & ~juegos['title'].str.contains(NO_JUEGO)]
    juegos['plataforma'] = juegos['cats'].map(plataforma)
    juegos['titulo'] = juegos['title'].map(titulo_base)
    juegos['clave'] = juegos['title'].map(clave)
    juegos['anio'] = juegos['details'].map(anio)
    juegos['n_calif'] = juegos['parent_asin'].map(ratings['parent_asin'].value_counts()).fillna(0)
    juegos = juegos[juegos['clave'].str.len() > 1]
    return juegos, ratings

In [9]:
juegos, ratings = cargar('datos/meta_juegos.csv.gz', 'datos/Video_Games.csv.gz')
print(f'Calificaciones originales: {len(ratings):,}')
print(f'Productos que son juegos: {len(juegos):,}, que se reducen a {juegos.clave.nunique():,} títulos al unir versiones')
juegos[['title', 'titulo', 'plataforma']].head(8)

Calificaciones originales: 814,586
Productos que son juegos: 11,112, que se reducen a 8,927 títulos al unir versiones


,title,titulo,plataforma
0,NBA 2K17 - Early Tip Off Edition - PlayStation 4,NBA 2K17,PlayStation 4
2,Konami Collector's Series: Castlevania & Contr...,Konami Collector's Series: Castlevania & Contra,PC
3,Tales of Vesperia - Definitive Edition - PlayS...,Tales of Vesperia,PlayStation 4
6,"Warhammer 40,000 Dawn of War Game of the Year ...","Warhammer 40,000 Dawn of War",PC
11,Zombi PlayStation 4,Zombi,PlayStation 4
12,Tom Clancy's Ghost Recon Advanced Warfighter,Tom Clancy's Ghost Recon Advanced Warfighter,PC
14,Wreckfest - Deluxe Edition - PlayStation 4,Wreckfest,PlayStation 4
16,Art of Murder: Cards of Destiny - PC,Art of Murder: Cards of Destiny,PC


## 2. Matriz usuario-juego

Se conservaron los juegos calificados por al menos 30 personas y las personas con al menos 5 juegos calificados, repitiendo el filtro hasta que ambas condiciones se cumplen a la vez. Si una persona calificó varias versiones del mismo juego se promedian.

In [10]:
def armar_matriz(juegos, ratings, min_usuarios_juego=40, min_juegos_usuario=5):
    """Une versiones por clave, promedia si un usuario calificó varias versiones y filtra."""
    r = ratings.merge(juegos[['parent_asin', 'clave']], on='parent_asin')
    r = r.groupby(['user_id', 'clave'], as_index=False)['rating'].mean()
    for _ in range(5):
        cj = r['clave'].value_counts()
        r = r[r['clave'].isin(cj[cj >= min_usuarios_juego].index)]
        cu = r['user_id'].value_counts()
        r = r[r['user_id'].isin(cu[cu >= min_juegos_usuario].index)]
    return r.reset_index(drop=True)


def soft_impute(M, observado, k=30, lam=5.0, iteraciones=60, tol=1e-4, semilla=SEMILLA):
    """Completa la matriz M (residuos centrados) donde observado es False.
    Cada iteración: rellena huecos con la reconstrucción, SVD de rango k y encoge los valores singulares en lam."""
    Z = np.zeros_like(M)
    for it in range(iteraciones):
        X = np.where(observado, M, Z)
        U, S, Vt = randomized_svd(X, n_components=k, random_state=semilla, n_iter=4)
        S = np.maximum(S - lam, 0)
        Z_nuevo = (U * S) @ Vt
        cambio = np.linalg.norm(Z_nuevo - Z) / (np.linalg.norm(Z) + 1e-9)
        Z = Z_nuevo
        if cambio < tol:
            break
    return U, S, Vt, it + 1


class Recomendador:
    """Línea base (promedio global + sesgo de usuario + sesgo de juego) más SoftImpute sobre los residuos."""

    def __init__(self, k=30, lam=5.0, reg_sesgo=5.0, alpha=1.0, usar_svd=True):
        self.k, self.lam, self.reg_sesgo, self.alpha, self.usar_svd = k, lam, reg_sesgo, alpha, usar_svd

    def fit(self, r):
        self.usuarios = pd.Index(r['user_id'].unique())
        self.juegos = pd.Index(r['clave'].unique())
        u = self.usuarios.get_indexer(r['user_id'])
        j = self.juegos.get_indexer(r['clave'])
        y = r['rating'].to_numpy(float)
        self.mu = y.mean()
        nu, nj = len(self.usuarios), len(self.juegos)
        bu, bj = np.zeros(nu), np.zeros(nj)
        for _ in range(10):
            bj = np.bincount(j, y - self.mu - bu[u], nj) / (np.bincount(j, minlength=nj) + self.reg_sesgo)
            bu = np.bincount(u, y - self.mu - bj[j], nu) / (np.bincount(u, minlength=nu) + self.reg_sesgo)
        self.bu, self.bj = bu, bj
        if self.usar_svd:
            M = np.zeros((nu, nj), dtype=np.float32)
            obs = np.zeros((nu, nj), dtype=bool)
            M[u, j] = y - self.mu - bu[u] - bj[j]
            obs[u, j] = True
            U, S, Vt, self.iteraciones = soft_impute(M, obs, self.k, self.lam)
            self.U, self.S, self.V = U, S, Vt.T
        return self

    def predecir(self, usuarios, claves):
        u = self.usuarios.get_indexer(usuarios)
        j = self.juegos.get_indexer(claves)
        p = self.mu + self.bu[u] + self.bj[j]
        if self.usar_svd:
            p = p + np.einsum('ik,ik->i', self.U[u] * self.S, self.V[j])
        return np.clip(p, 1, 5)

    def perfil_nuevo(self, claves, calif):
        """Lo que hace la app: ubica a un usuario nuevo a partir de sus calificaciones."""
        j = self.juegos.get_indexer(claves)
        y = np.asarray(calif, float)
        bu = np.sum(y - self.mu - self.bj[j]) / (len(y) + self.reg_sesgo)
        w = np.zeros(len(self.S))
        if self.usar_svd:
            B = self.V[j]
            resid = y - self.mu - bu - self.bj[j]
            w = np.linalg.solve(B.T @ B + self.alpha * np.eye(B.shape[1]), B.T @ resid)
        return bu, w

    def predecir_nuevo(self, bu, w):
        p = self.mu + bu + self.bj
        if self.usar_svd:
            p = p + self.V @ w
        return np.clip(p, 1, 5)

In [11]:
r = armar_matriz(juegos, ratings, min_usuarios_juego=30, min_juegos_usuario=5)
nu, nj = r.user_id.nunique(), r.clave.nunique()
print(f'Personas: {nu:,}   juegos: {nj:,}   calificaciones: {len(r):,}')
print(f'Casillas con calificación: {100 * len(r) / (nu * nj):.2f}%')
print('Distribución de estrellas:')
print((r.rating.round().value_counts(normalize=True).sort_index() * 100).round(1).to_string())

Personas: 17,035   juegos: 1,433   calificaciones: 144,652
Casillas con calificación: 0.59%
Distribución de estrellas:
rating
1.0     5.6
2.0     4.7
3.0     9.9
4.0    18.4
5.0    61.3


La matriz tiene menos del 1% de las casillas llenas y la mayoría de las calificaciones son de 5 estrellas. Esto último resultó determinante para elegir el modelo.

## 3. Comparación de métodos

Para simular lo que hace la app, se separó el 20% de las personas y el modelo se entrenó sin ellas. A cada persona de prueba se le escondió un juego que calificó con 4 o 5 estrellas; con el resto de sus calificaciones se le ubicó en el modelo y se revisó si el juego escondido aparecía entre sus 10 primeras recomendaciones.

Se compararon cuatro opciones:

- **Al azar.**
- **Los más populares:** la misma lista para todos.
- **SoftImpute:** completar la matriz prediciendo las estrellas, como se vio en clase.
- **PCA de la matriz:** descomposición en valores singulares de la matriz con los huecos en 0, dividiendo cada columna entre la raíz del número de personas que calificaron el juego para que los muy populares no dominen. Al ubicar a una persona, cada calificación pesa estrellas − 2: cinco estrellas suman 3 y una estrella resta 1.

In [12]:
rng = np.random.RandomState(SEMILLA)
usuarios = r.user_id.unique()
prueba_u = set(rng.choice(usuarios, size=int(0.2 * len(usuarios)), replace=False))
tr = r[~r.user_id.isin(prueba_u)]
te = r[r.user_id.isin(prueba_u)]
claves = pd.Index(sorted(tr.clave.unique()))
te = te[te.clave.isin(claves)]

casos = []
for u, g in te.groupby('user_id'):
    gustados = g[g.rating >= 4]
    if len(g) < 5 or len(gustados) == 0:
        continue
    objetivo = gustados.sample(1, random_state=SEMILLA).iloc[0]
    resto = g[g.clave != objetivo.clave]
    casos.append((resto.clave.tolist(), claves.get_indexer(resto.clave), resto.rating.to_numpy(), claves.get_loc(objetivo.clave)))
print(f'Personas de prueba evaluadas: {len(casos):,}')

def acierto_top10(fn):
    aciertos = 0
    for cl, idx, calif, obj in casos:
        s = fn(cl, idx, calif).astype(float).copy()
        s[idx] = -np.inf
        aciertos += (s > s[obj]).sum() < 10
    return aciertos / len(casos)

U = pd.Index(tr.user_id.unique())
ji = claves.get_indexer(tr.clave)
pop = np.bincount(ji, minlength=len(claves)).astype(float)
resultados = {}
rng2 = np.random.RandomState(SEMILLA)
resultados['Al azar'] = acierto_top10(lambda cl, i, c: rng2.rand(len(claves)))
resultados['Los más populares'] = acierto_top10(lambda cl, i, c: pop)

si = Recomendador(k=40, lam=10, alpha=0.1).fit(tr)
orden = si.juegos.get_indexer(claves)
def soft_impute_fn(cl, i, c):
    bu, w = si.perfil_nuevo(cl, c)
    return si.predecir_nuevo(bu, w)[orden]
resultados['SoftImpute (predecir estrellas)'] = acierto_top10(soft_impute_fn)

X = sp.csr_matrix((tr.rating.to_numpy(float), (U.get_indexer(tr.user_id), ji)), shape=(len(U), len(claves)))
norma = np.sqrt(pop)
_, S, Vt = randomized_svd(X @ sp.diags(1 / norma), n_components=50, random_state=SEMILLA, n_iter=5)
Vp = Vt.T
resultados['PCA de la matriz (50 componentes)'] = acierto_top10(
    lambda cl, i, c: (Vp[i].T @ ((c - 2.0) / norma[i])) @ Vp.T * norma)

pd.DataFrame({'El juego escondido aparece en el top 10 (%)': {k: round(100 * v, 1) for k, v in resultados.items()}})

Personas de prueba evaluadas: 3,377


,El juego escondido aparece en el top 10 (%)
Al azar,0.7
Los más populares,8.1
SoftImpute (predecir estrellas),4.9
PCA de la matriz (50 componentes),19.1


SoftImpute predice bien el número de estrellas en promedio, pero casi no sirve para ordenar recomendaciones: como la mayoría de las calificaciones son de 5 estrellas, el número de estrellas dice poco. Lo que sí dice mucho es qué juegos calificó cada persona, y eso es lo que aprovecha el PCA de la matriz. Por eso fue el modelo elegido: acierta más del doble que recomendar lo más popular.

## 4. Modelo final y exportación

El modelo final se entrena con todas las personas y se exporta a `../data/juegos.js`. Los 50 componentes de cada juego se guardan como enteros de 16 bits para que el archivo pese poco; el error que introduce esa compresión es despreciable. También se imprimen los juegos más extremos de cada componente, que sirvieron para ponerle nombre a los ejes del mapa.

In [13]:
K = 50
SEMILLA = 69
SALIDA = '../data/juegos.js'

CORTO = {
    'Nintendo Switch': 'Switch', 'PlayStation 5': 'PS5', 'PlayStation 4': 'PS4', 'PlayStation 3': 'PS3',
    'PlayStation 2': 'PS2', 'PlayStation': 'PS1', 'PlayStation Vita': 'PS Vita', 'Sony PSP': 'PSP',
    'PlayStation Digital Content': 'PlayStation', 'Xbox Series X & S': 'Xbox Series', 'Xbox One': 'Xbox One',
    'Xbox 360': 'Xbox 360', 'Xbox': 'Xbox', 'Xbox Digital Content': 'Xbox', 'Nintendo 3DS & 2DS': '3DS',
    'Nintendo DS': 'DS', 'Nintendo 64': 'N64', 'Super Nintendo': 'SNES', 'Nintendo NES': 'NES',
    'Game Boy Advance': 'GBA', 'Game Boy Color': 'GBC', 'Game Boy': 'Game Boy', 'Wii U': 'Wii U', 'Wii': 'Wii',
    'GameCube': 'GameCube', 'PC': 'PC', 'Mac': 'Mac', 'Sega Genesis': 'Genesis', 'Sega Dreamcast': 'Dreamcast',
    'Sega Saturn': 'Saturn',
}
ORDEN_PLAT = list(CORTO.values())


def familia(plataforma):
    p = plataforma.lower()
    if any(s in p for s in ('nintendo', 'wii', 'gamecube', 'game boy', 'switch')):
        return 'nintendo'
    if any(s in p for s in ('playstation', 'psp')):
        return 'playstation'
    if 'xbox' in p:
        return 'xbox'
    if p in ('pc', 'mac'):
        return 'pc'
    return 'otra'


juegos, ratings = cargar('datos/meta_juegos.csv.gz', 'datos/Video_Games.csv.gz')
r = armar_matriz(juegos, ratings, 30, 5)

# Catálogo ordenado por número de usuarios (más jugados primero)
conteo = r['clave'].value_counts()
claves = pd.Index(conteo.index)
usuarios = pd.Index(r['user_id'].unique())
X = sp.csr_matrix((r['rating'].to_numpy(float),
                   (usuarios.get_indexer(r['user_id']), claves.get_indexer(r['clave']))),
                  shape=(len(usuarios), len(claves)))
n = conteo.to_numpy(float)
norma = np.sqrt(n)
Xn = X @ sp.diags(1 / norma)
_, S, Vt = randomized_svd(Xn, n_components=K, random_state=SEMILLA, n_iter=7)
V = Vt.T
pve = S ** 2 / Xn.multiply(Xn).sum()

# Metadatos por juego: la versión más calificada da título e imagen; se juntan plataformas y año más antiguo
versiones = juegos[juegos['clave'].isin(claves)].sort_values('n_calif', ascending=False)
filas = []
for c in claves:
    g = versiones[versiones['clave'] == c]
    principal = g.iloc[0]
    plats = sorted({CORTO.get(p, p) for p in g['plataforma'] if p != 'Otra'},
                   key=lambda p: ORDEN_PLAT.index(p) if p in ORDEN_PLAT else 99)
    anio = g['anio'].min()
    filas.append({
        't': principal['titulo'],
        'p': plats,
        'f': familia(principal['plataforma']),
        'y': None if pd.isna(anio) else int(anio),
        'i': principal['image'] if isinstance(principal['image'], str) else None,
        'n': int(conteo[c]),
    })

# Vectores a enteros de 16 bits
escala = 32000 / np.abs(V).max()
Vq = np.round(V * escala).astype('<i2')
error_cuant = np.abs(Vq / escala - V).max()

# Muestra real de la matriz para la explicación: 24 personas más activas y 36 juegos más calificados
top_juegos = claves[:36]
sub = r[r['clave'].isin(top_juegos)]
top_usuarios = sub['user_id'].value_counts().index[:24]
sub = sub[sub['user_id'].isin(top_usuarios)]
tabla = sub.pivot_table(index='user_id', columns='clave', values='rating').reindex(index=top_usuarios, columns=top_juegos)
muestra = [[0 if pd.isna(v) else int(round(v)) for v in fila] for fila in tabla.to_numpy()]
llenas = sum(v > 0 for fila in muestra for v in fila)
print(f'Muestra: {llenas} de {24 * 36} casillas con calificación')

datos = {
    'k': K,
    'muestra': muestra,
    'escala': float(escala),
    'vectores': base64.b64encode(Vq.tobytes()).decode('ascii'),
    'pve': [round(float(p), 5) for p in pve],
    'usuarios': int(len(usuarios)),
    'calificaciones': int(len(r)),
    'juegos': filas,
}
with open(SALIDA, 'w', encoding='utf-8') as f:
    f.write('// Generado por exportar.py. Modelo: PCA (SVD) de la matriz usuario-juego, k = 50, semilla 69.\n')
    f.write('window.DATOS = ')
    json.dump(datos, f, ensure_ascii=False, separators=(',', ':'))
    f.write(';\n')

print(f'Juegos: {len(filas)}  usuarios: {len(usuarios)}  calificaciones: {len(r)}')
print(f'Error máximo por cuantizar: {error_cuant:.2e} (valores típicos {np.abs(V).mean():.2e})')
print(f'Varianza explicada por los {K} componentes: {pve.sum():.3f}')
print('Familias:', pd.Series([x["f"] for x in filas]).value_counts().to_dict())
print('Sin año:', sum(x['y'] is None for x in filas), ' sin imagen:', sum(x['i'] is None for x in filas))

# Para etiquetar los ejes del mapa: juegos más extremos de cada componente
titulos = np.array([x['t'] for x in filas])
for k in range(8):
    o = np.argsort(V[:, k])
    print(f'\nC{k + 1} (+): ' + ' | '.join(titulos[o[::-1][:10]]))
    print(f'C{k + 1} (-): ' + ' | '.join(titulos[o[:10]]))
    fam = pd.Series([x['f'] for x in filas])
    print('   familia promedio +10%:', fam[o[::-1][:140]].value_counts().head(3).to_dict(),
          ' -10%:', fam[o[:140]].value_counts().head(3).to_dict())

Muestra: 333 de 864 casillas con calificación
Juegos: 1433  usuarios: 17035  calificaciones: 144652
Error máximo por cuantizar: 5.93e-06 (valores típicos 1.95e-02)
Varianza explicada por los 50 componentes: 0.114
Familias: {'playstation': 607, 'xbox': 352, 'nintendo': 330, 'pc': 141, 'otra': 3}
Sin año: 17  sin imagen: 0

C1 (+): Resident Evil 4 | Tomb Raider | Resident Evil 5 | Batman: Arkham Asylum | Skyrim VR | Uncharted 2: Among Thieves | Grand Theft Auto V | Assassin's Creed IV Black Flag | Metal Gear Solid 4: Guns of the Patriots | Resident Evil: Revelations
C1 (-): The Sims 3: Late Night | The Sims 3: World Adventures Expansion Pack | The Sims 3: Generations | Madden NFL 21 | The Sims 3 Island Paradise | The Sims 3: Master Suite Stuff | Rollercoaster Tycoon 3: Platinum | Madden NFL 25 Anniversary Edition with NFL Sunday Ticket | Spore | The Sims 3: Ambitions
   familia promedio +10%: {'playstation': 70, 'nintendo': 33, 'xbox': 27}  -10%: {'playstation': 52, 'xbox': 38, 'pc': 30}